## Trạng thái bản lưu trên GitHub

Baseline lịch sử; không so sánh như ablation cùng recipe của notebook 07.

Output cell đã được bỏ để tránh file tạm, dữ liệu nhạy cảm và bảng lỗi cũ. Mã cell được giữ; kết quả kiểm chứng nằm trong thư mục `results/`. Xem `docs/notebook-guide.md` trước khi chạy. Không cần chạy tuần tự cả tám notebook.

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


def find_kaggle_file(filename):
    """Tìm file trong phiên hiện tại hoặc Notebook Output đã gắn."""
    working_path = Path("/kaggle/working") / filename

    if working_path.exists():
        return working_path

    candidates = sorted(
        Path("/kaggle/input").rglob(filename)
    )

    if not candidates:
        raise FileNotFoundError(
            f"Không tìm thấy {filename}"
        )

    if len(candidates) > 1:
        print(
            f"Tìm thấy nhiều bản {filename}, "
            f"đang dùng: {candidates[0]}"
        )

    return candidates[0]


FEATURE_PATH = find_kaggle_file(
    "gmdcsa24_features_32.npz"
)

MANIFEST_PATH = find_kaggle_file(
    "gmdcsa24_pose_manifest.csv"
)


# Đọc các đặc trưng đã tạo
with np.load(FEATURE_PATH) as data:
    motion_features = data[
        "motion_features"
    ].astype(np.float32)

    global_features = data[
        "global_features"
    ].astype(np.float32)

    body_scales = data[
        "body_scales"
    ].astype(np.float32)

    labels = data[
        "labels"
    ].astype(np.int64)

    subjects = data[
        "subjects"
    ].astype(np.int64)

    video_ids = data[
        "video_ids"
    ].astype("U")


pose_manifest = pd.read_csv(MANIFEST_PATH)


# Căn manifest theo đúng thứ tự trong NPZ
pose_manifest = (
    pose_manifest
    .set_index("video_id")
    .loc[video_ids]
    .reset_index()
)


assert len(video_ids) == 160
assert np.array_equal(
    pose_manifest["subject"].to_numpy(),
    subjects,
)
assert np.isfinite(motion_features).all()
assert np.isfinite(global_features).all()


print("Feature file:", FEATURE_PATH)
print("Manifest file:", MANIFEST_PATH)
print()
print("Motion:", motion_features.shape)
print("Global:", global_features.shape)
print("Labels:", labels.shape)
print("Subjects:", sorted(np.unique(subjects)))
print("Số video:", len(video_ids))

In [ ]:
nested_records = []
protocol_records = []

all_subjects = sorted(
    np.unique(subjects).tolist()
)


for test_subject in all_subjects:
    # Ba subject còn lại tham gia vòng trong
    development_subjects = [
        subject
        for subject in all_subjects
        if subject != test_subject
    ]

    for val_subject in development_subjects:
        train_subjects = [
            subject
            for subject in development_subjects
            if subject != val_subject
        ]

        protocol_records.append({
            "outer_test": f"S{test_subject}",
            "inner_validation": f"S{val_subject}",
            "inner_train": ", ".join(
                f"S{subject}"
                for subject in train_subjects
            ),
            "final_train": ", ".join(
                f"S{subject}"
                for subject in development_subjects
            ),
        })

        for sample_index in range(len(video_ids)):
            current_subject = int(
                subjects[sample_index]
            )

            if current_subject == test_subject:
                split = "test"
            elif current_subject == val_subject:
                split = "val"
            else:
                split = "train"

            nested_records.append({
                "outer_fold": (
                    f"test_subject_{test_subject}"
                ),
                "inner_fold": (
                    f"val_subject_{val_subject}"
                ),
                "sample_index": sample_index,
                "video_id": video_ids[sample_index],
                "subject": current_subject,
                "label_id": int(
                    labels[sample_index]
                ),
                "split": split,
                "test_subject": test_subject,
                "val_subject": val_subject,
                "train_subjects": ",".join(
                    map(str, train_subjects)
                ),
            })


nested_folds = pd.DataFrame(
    nested_records
)

protocol_summary = pd.DataFrame(
    protocol_records
)


# 4 vòng ngoài × 3 vòng trong × 160 video
assert len(nested_folds) == 4 * 3 * 160
assert nested_folds["outer_fold"].nunique() == 4

NESTED_FOLDS_PATH = Path(
    "/kaggle/working/"
    "gmdcsa24_nested_loso_folds.csv"
)

nested_folds.to_csv(
    NESTED_FOLDS_PATH,
    index=False,
)


display(protocol_summary)

print("Số outer folds:", 4)
print("Số inner folds:", 12)
print("Số dòng:", len(nested_folds))
print("Đã lưu:", NESTED_FOLDS_PATH)

In [ ]:
for (
    outer_fold,
    inner_fold,
), fold_data in nested_folds.groupby(
    ["outer_fold", "inner_fold"]
):
    train_subject_set = set(
        fold_data.loc[
            fold_data["split"] == "train",
            "subject",
        ]
    )

    val_subject_set = set(
        fold_data.loc[
            fold_data["split"] == "val",
            "subject",
        ]
    )

    test_subject_set = set(
        fold_data.loc[
            fold_data["split"] == "test",
            "subject",
        ]
    )

    # Train, validation và test không được chung người
    assert train_subject_set.isdisjoint(
        val_subject_set
    )
    assert train_subject_set.isdisjoint(
        test_subject_set
    )
    assert val_subject_set.isdisjoint(
        test_subject_set
    )

    assert len(train_subject_set) == 2
    assert len(val_subject_set) == 1
    assert len(test_subject_set) == 1


print("Nested LOSO hợp lệ.")
print("Không có subject leakage.")

In [ ]:
num_samples = len(video_ids)
num_joints = motion_features.shape[2]


# Hai kênh đầu là vận tốc x và y
joint_velocity_xy = motion_features[
    ..., :2
]


# Thống kê vận tốc có hướng của từng khớp
velocity_mean = joint_velocity_xy.mean(
    axis=1
).reshape(num_samples, -1)

velocity_std = joint_velocity_xy.std(
    axis=1
).reshape(num_samples, -1)

velocity_max_abs = np.abs(
    joint_velocity_xy
).max(axis=1).reshape(num_samples, -1)


# Độ lớn vận tốc của từng khớp
joint_speed = np.linalg.norm(
    joint_velocity_xy,
    axis=-1,
)

speed_mean = joint_speed.mean(axis=1)
speed_std = joint_speed.std(axis=1)
speed_max = joint_speed.max(axis=1)
speed_p95 = np.quantile(
    joint_speed,
    0.95,
    axis=1,
)


# Góc nghiêng thân nằm ở kênh số 9
torso_angle = global_features[..., 9]

torso_features = np.column_stack([
    torso_angle.mean(axis=1),
    torso_angle.std(axis=1),
    torso_angle.min(axis=1),
    torso_angle.max(axis=1),
    np.ptp(torso_angle, axis=1),
    torso_angle[:, -1] - torso_angle[:, 0],
])


# Tâm hông x, y
hip_xy = global_features[..., :2]

safe_body_scale = np.maximum(
    body_scales,
    1e-6,
)


# Độ dịch chuyển so với frame đầu
hip_displacement = (
    hip_xy - hip_xy[:, :1]
) / safe_body_scale[:, None, None]

hip_radius = np.linalg.norm(
    hip_displacement,
    axis=-1,
)

hip_step_distance = np.linalg.norm(
    np.diff(hip_displacement, axis=1),
    axis=-1,
)

hip_features = np.column_stack([
    hip_displacement[:, -1, 0],
    hip_displacement[:, -1, 1],

    hip_displacement[:, :, 0].min(axis=1),
    hip_displacement[:, :, 1].min(axis=1),

    hip_displacement[:, :, 0].max(axis=1),
    hip_displacement[:, :, 1].max(axis=1),

    np.ptp(hip_displacement[:, :, 0], axis=1),
    np.ptp(hip_displacement[:, :, 1], axis=1),

    hip_radius.max(axis=1),
    hip_radius[:, -1],
    hip_step_distance.sum(axis=1),
])


# Chiều cao cơ thể nằm ở kênh số 11
body_height = global_features[..., 11]

initial_height = np.maximum(
    np.abs(body_height[:, :1]),
    1e-6,
)

height_change = (
    body_height - body_height[:, :1]
) / initial_height

height_features = np.column_stack([
    height_change.mean(axis=1),
    height_change.std(axis=1),
    height_change.min(axis=1),
    height_change.max(axis=1),
    np.ptp(height_change, axis=1),
    height_change[:, -1],
])


# Ghép thành một vector cố định cho mỗi video
X_motion_svm = np.concatenate([
    velocity_mean,
    velocity_std,
    velocity_max_abs,
    speed_mean,
    speed_std,
    speed_max,
    speed_p95,
    torso_features,
    hip_features,
    height_features,
], axis=1).astype(np.float32)


assert X_motion_svm.shape[0] == 160
assert np.isfinite(X_motion_svm).all()


print(
    "Kích thước đặc trưng Motion-SVM:",
    X_motion_svm.shape,
)

print(
    "Số đặc trưng mỗi video:",
    X_motion_svm.shape[1],
)

print(
    "Finite:",
    np.isfinite(X_motion_svm).all(),
)

In [ ]:
MOTION_SVM_FEATURE_PATH = Path(
    "/kaggle/working/"
    "gmdcsa24_motion_svm_features.npz"
)


np.savez_compressed(
    MOTION_SVM_FEATURE_PATH,
    features=X_motion_svm,
    labels=labels,
    subjects=subjects,
    video_ids=video_ids,
)


print("Đã lưu:", MOTION_SVM_FEATURE_PATH)
print(
    "Dung lượng:",
    round(
        MOTION_SVM_FEATURE_PATH.stat().st_size
        / 1024**2,
        2,
    ),
    "MB",
)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)


def create_svm_model(C, gamma):
    return Pipeline([
        (
            "scaler",
            StandardScaler(),
        ),
        (
            "svm",
            SVC(
                kernel="rbf",
                C=C,
                gamma=gamma,
                probability=False,
                cache_size=1000,
            ),
        ),
    ])


def calculate_metrics(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1],
    ).ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else 0.0
    )

    false_alarm_rate = (
        fp / (fp + tn)
        if (fp + tn) > 0
        else 0.0
    )

    return {
        "accuracy": accuracy_score(
            y_true,
            y_pred,
        ),
        "balanced_accuracy": (
            balanced_accuracy_score(
                y_true,
                y_pred,
            )
        ),
        "precision": precision_score(
            y_true,
            y_pred,
            zero_division=0,
        ),
        "sensitivity": recall_score(
            y_true,
            y_pred,
            zero_division=0,
        ),
        "specificity": specificity,
        "f1": f1_score(
            y_true,
            y_pred,
            zero_division=0,
        ),
        "false_alarm_rate": false_alarm_rate,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


# 16 cấu hình, mỗi outer fold kiểm tra 3 inner folds
PARAMETER_GRID = [
    {
        "C": C,
        "gamma": gamma,
    }
    for C in [0.1, 1, 10, 100]
    for gamma in [
        "scale",
        0.001,
        0.01,
        0.1,
    ]
]


print(
    "Số cấu hình cần thử:",
    len(PARAMETER_GRID),
)


In [ ]:
import joblib
from pathlib import Path


MODEL_DIR = Path(
    "/kaggle/working/motion_svm_models"
)

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


inner_result_records = []
outer_result_records = []
prediction_records = []


for test_subject in all_subjects:
    development_subjects = [
        subject
        for subject in all_subjects
        if subject != test_subject
    ]

    candidate_summaries = []

    print(
        f"\n===== OUTER TEST: S{test_subject} ====="
    )

    # Vòng trong: chọn C và gamma
    for candidate_id, params in enumerate(
        PARAMETER_GRID
    ):
        validation_metrics = []

        for val_subject in development_subjects:
            train_subjects = [
                subject
                for subject in development_subjects
                if subject != val_subject
            ]

            train_mask = np.isin(
                subjects,
                train_subjects,
            )

            val_mask = (
                subjects == val_subject
            )

            model = create_svm_model(
                C=params["C"],
                gamma=params["gamma"],
            )

            model.fit(
                X_motion_svm[train_mask],
                labels[train_mask],
            )

            val_prediction = model.predict(
                X_motion_svm[val_mask]
            )

            metrics = calculate_metrics(
                labels[val_mask],
                val_prediction,
            )

            validation_metrics.append(metrics)

            inner_result_records.append({
                "test_subject": test_subject,
                "val_subject": val_subject,
                "train_subjects": ",".join(
                    map(str, train_subjects)
                ),
                "candidate_id": candidate_id,
                "C": params["C"],
                "gamma": str(params["gamma"]),
                **metrics,
            })

        candidate_summaries.append({
            "candidate_id": candidate_id,
            "C": params["C"],
            "gamma": str(params["gamma"]),
            "mean_balanced_accuracy": np.mean([
                item["balanced_accuracy"]
                for item in validation_metrics
            ]),
            "mean_f1": np.mean([
                item["f1"]
                for item in validation_metrics
            ]),
            "mean_sensitivity": np.mean([
                item["sensitivity"]
                for item in validation_metrics
            ]),
        })

    candidate_table = pd.DataFrame(
        candidate_summaries
    )

    # Ưu tiên balanced accuracy, sau đó F1 và sensitivity
    candidate_table = candidate_table.sort_values(
        [
            "mean_balanced_accuracy",
            "mean_f1",
            "mean_sensitivity",
            "C",
            "candidate_id",
        ],
        ascending=[
            False,
            False,
            False,
            True,
            True,
        ],
        kind="mergesort",
    )

    best_candidate_id = int(
        candidate_table.iloc[0]["candidate_id"]
    )

    best_params = PARAMETER_GRID[
        best_candidate_id
    ]

    print(
        "Best C:",
        best_params["C"],
        "| Best gamma:",
        best_params["gamma"],
    )

    print(
        "Inner balanced accuracy:",
        round(
            candidate_table.iloc[0][
                "mean_balanced_accuracy"
            ],
            4,
        ),
    )

    # Train lại trên toàn bộ ba subject còn lại
    final_train_mask = (
        subjects != test_subject
    )

    test_mask = (
        subjects == test_subject
    )

    final_model = create_svm_model(
        C=best_params["C"],
        gamma=best_params["gamma"],
    )

    final_model.fit(
        X_motion_svm[final_train_mask],
        labels[final_train_mask],
    )

    test_prediction = final_model.predict(
        X_motion_svm[test_mask]
    )

    decision_scores = final_model.decision_function(
        X_motion_svm[test_mask]
    )

    test_metrics = calculate_metrics(
        labels[test_mask],
        test_prediction,
    )

    outer_result_records.append({
        "test_subject": test_subject,
        "train_subjects": ",".join(
            map(str, development_subjects)
        ),
        "C": best_params["C"],
        "gamma": str(best_params["gamma"]),
        "train_samples": int(
            final_train_mask.sum()
        ),
        "test_samples": int(
            test_mask.sum()
        ),
        **test_metrics,
    })

    test_indices = np.where(test_mask)[0]

    for local_index, sample_index in enumerate(
        test_indices
    ):
        prediction_records.append({
            "sample_index": int(sample_index),
            "video_id": video_ids[sample_index],
            "subject": int(subjects[sample_index]),
            "true_label": int(
                labels[sample_index]
            ),
            "predicted_label": int(
                test_prediction[local_index]
            ),
            "decision_score": float(
                decision_scores[local_index]
            ),
            "correct": bool(
                test_prediction[local_index]
                == labels[sample_index]
            ),
        })

    model_path = (
        MODEL_DIR
        / f"motion_svm_test_subject_{test_subject}.joblib"
    )

    joblib.dump(
        final_model,
        model_path,
    )

    print(
        "Test balanced accuracy:",
        round(
            test_metrics["balanced_accuracy"],
            4,
        ),
    )

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay


inner_results = pd.DataFrame(
    inner_result_records
)

outer_results = pd.DataFrame(
    outer_result_records
)

predictions = (
    pd.DataFrame(prediction_records)
    .sort_values("sample_index")
    .reset_index(drop=True)
)


assert len(predictions) == 160
assert predictions["sample_index"].nunique() == 160


metric_columns = [
    "accuracy",
    "balanced_accuracy",
    "precision",
    "sensitivity",
    "specificity",
    "f1",
    "false_alarm_rate",
]


metric_summary = (
    outer_results[metric_columns]
    .agg(["mean", "std"])
    .T
    .reset_index()
    .rename(columns={
        "index": "metric",
    })
)


overall_metrics = calculate_metrics(
    predictions["true_label"],
    predictions["predicted_label"],
)


print("KẾT QUẢ THEO OUTER FOLD")
display(
    outer_results.round(4)
)


print("\nTRUNG BÌNH 4 OUTER FOLDS")
display(
    metric_summary.round(4)
)


print("\nKẾT QUẢ GỘP 160 VIDEO")

for metric_name in metric_columns:
    print(
        f"{metric_name}:",
        round(
            overall_metrics[metric_name],
            4,
        ),
    )


# Confusion matrix gộp toàn bộ outer test
matrix = confusion_matrix(
    predictions["true_label"],
    predictions["predicted_label"],
    labels=[0, 1],
)

display_object = ConfusionMatrixDisplay(
    confusion_matrix=matrix,
    display_labels=["ADL", "FALL"],
)

display_object.plot(
    cmap="Blues",
    values_format="d",
)

plt.title(
    "Motion-SVM — Nested LOSO"
)

plt.tight_layout()

CONFUSION_PATH = Path(
    "/kaggle/working/"
    "gmdcsa24_motion_svm_confusion_matrix.png"
)

plt.savefig(
    CONFUSION_PATH,
    dpi=200,
    bbox_inches="tight",
)

plt.show()


# Lưu các bảng kết quả
inner_results.to_csv(
    "/kaggle/working/"
    "gmdcsa24_motion_svm_inner_results.csv",
    index=False,
)

outer_results.to_csv(
    "/kaggle/working/"
    "gmdcsa24_motion_svm_outer_results.csv",
    index=False,
)

predictions.to_csv(
    "/kaggle/working/"
    "gmdcsa24_motion_svm_predictions.csv",
    index=False,
)

metric_summary.to_csv(
    "/kaggle/working/"
    "gmdcsa24_motion_svm_summary.csv",
    index=False,
)


print("\nĐã lưu toàn bộ kết quả Motion-SVM.")

In [ ]:
# Xem kết quả riêng trên từng subject
columns_to_show = [
    "test_subject",
    "C",
    "gamma",
    "test_samples",
    "accuracy",
    "balanced_accuracy",
    "precision",
    "sensitivity",
    "specificity",
    "f1",
    "false_alarm_rate",
    "fp",
    "fn",
]

display(
    outer_results[columns_to_show]
    .round(4)
)


# Liệt kê 27 video dự đoán sai
error_videos = predictions.loc[
    ~predictions["correct"]
].copy()

display(error_videos)

print("Tổng video sai:", len(error_videos))

print("\nSố lỗi theo subject:")
print(
    error_videos.groupby(
        ["subject", "true_label", "predicted_label"]
    ).size()
)